In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Shared telescope geometry, the danish-based donut simulator, and figure
# helpers live in the installed `shape_vs_intensity` package (pip install -e .).
import galsim.zernike as gzk
from shape_vs_intensity import config as C
from shape_vs_intensity import sim
from shape_vs_intensity.plotutils import use_style

use_style()

In [ ]:
DEFOCAL_TYPE = "intra"

In [ ]:
DIAGRAM_L = 1.25
# Pixel radius matching the same paraxial/Danish normalization used by map_circles.
RING_RADIUS_PIX = C.DEFOCAL_OFFSET / (2 * C.FOCAL_RATIO) / C.PIXEL_SCALE


def simulate_donut(ax, zernikes, defocal_type="intra"):
    """Simulate a donut image with the given aberrations

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes on which to draw the circles.
    zernikes : dict
        Aberration Zernike amplitudes, as ``{noll_index: coefficient_in_meters}``
    defocal_type : str
        "intra" or "extra".
    """
    img = sim.simulate_donut(zernikes=zernikes, defocal_type=defocal_type)

    half_extent = (C.NPIX // 2 + 0.5) / RING_RADIUS_PIX
    ax.imshow(
        img,
        origin="lower",
        extent=(-half_extent, +half_extent, -half_extent, +half_extent),
    )
    ax.set(
        xlim=(-DIAGRAM_L, +DIAGRAM_L),
        ylim=(-DIAGRAM_L, +DIAGRAM_L),
        aspect="equal",
        xticks=[],
        yticks=[],
    )


def map_circles(ax, zernikes, defocal_type="intra", N=5):
    """Map concentric pupil circles through a displacement field.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes on which to draw the circles.
    zernikes : dict
        Aberration Zernike amplitudes, as ``{noll_index: coefficient_in_meters}``
    defocal_type : str
        "intra" or "extra".
    N : int, optional
        Number of pupil circles to draw.
    """
    displacement = zernike_displacement(zernikes)

    defocal_sign = (
        C.INTRA_DEFOCUS_SIGN if defocal_type == "intra" else C.EXTRA_DEFOCUS_SIGN
    )

    theta = np.linspace(0, 2 * np.pi, 10_000)
    for r in np.linspace(C.EPS, 1, N):
        x0 = -defocal_sign * r * np.cos(theta)
        y0 = -defocal_sign * r * np.sin(theta)
        dx, dy = displacement(r, theta)
        ax.plot(x0 + dx, y0 + dy, c="C1", lw=0.5)

    ax.set(
        xlim=(-DIAGRAM_L, +DIAGRAM_L),
        ylim=(-DIAGRAM_L, +DIAGRAM_L),
        aspect="equal",
        xticks=[],
        yticks=[],
    )


# Calibrate ring displacements to wavefront coefficients (meters)
RING_SCALE = 4 * C.FOCAL_RATIO**2 / C.DEFOCAL_OFFSET


def zernike_displacement(zernikes):
    """Return ring displacement for the given aberrations.

    Parameters
    ----------
    zernikes : dict
        Aberration Zernike amplitudes, as ``{noll_index: coefficient_in_meters}``

    Returns
    -------
    callable
        ``displacement(rho, theta) -> (dx, dy)`` in donut-radius units per meter
        of Zernike coefficient.
    """
    coeff = np.zeros(max(zernikes.keys()) + 1)
    for j, amp in zernikes.items():
        coeff[j] = amp
    zern = gzk.Zernike(coeff, R_outer=1.0, R_inner=C.EPS)
    grad_x, grad_y = zern.gradX, zern.gradY

    def displacement(rho, theta):
        x = rho * np.cos(theta)
        y = rho * np.sin(theta)
        return -RING_SCALE * grad_x(x, y), -RING_SCALE * grad_y(x, y)

    return displacement

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Defocus\n$\\nu = 0,\\, m=0$")
axes[0, 1].set_title("Tilt\n$\\nu = 0,\\, m=1$")

for col, zernikes in enumerate([{4: 0.8e-5}, {2: 3e-5}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_defocus_tilt.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("Astigmatism\n$\\nu = 0,\\, m=2$")
axes[0, 1].set_title("Trefoil\n$\\nu = 0,\\, m=3$")
axes[0, 2].set_title("Quadrafoil\n$\\nu = 0,\\, m=4$")
axes[0, 3].set_title("Pentafoil\n$\\nu = 0,\\, m=5$")

for col, zernikes in enumerate([{6: 7e-6}, {10: 2e-6}, {14: 1e-6}, {20: 1e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Spherical\n$\\nu = 1,\\, m=0$")
axes[0, 1].set_title("Coma\n$\\nu = 1,\\, m=1$")

for col, zernikes in enumerate([{11: 3e-7}, {8: 2e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_spherical_coma.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("2nd Astigmatism\n$\\nu = 1,\\, m=2$")
axes[0, 1].set_title("2nd Trefoil\n$\\nu = 1,\\, m=3$")
axes[0, 2].set_title("2nd Quadrafoil\n$\\nu = 1,\\, m=4$")
axes[0, 3].set_title("2nd Pentafoil\n$\\nu = 1,\\, m=5$")

for col, zernikes in enumerate(
    [{12: 0.75e-6}, {18: 0.4e-6}, {26: 0.22e-6}, {34: 0.12e-6}]
):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_2nd_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("2nd Spherical\n$\\nu = 2,\\, m=0$")
axes[0, 1].set_title("2nd Coma\n$\\nu = 2,\\, m=1$")

for col, zernikes in enumerate([{22: 0.10e-6}, {16: 0.2e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_2nd_spherical_coma.pdf", bbox_inches="tight")